# Metoda Ant Colony Optimization pro úlohu O-CVRPTW

Tento notebook implementuje metodu Ant Colony Optimization (ACO) adaptovanou na selektivní kapacitní rozvozní úlohu s časovými okny (O-CVRPTW) dle kapitoly 1.6 diplomové práce. Metoda je konstruktivní, řízená feromonovou pamětí a heuristickou informací.

**Struktura notebooku:**
1. Konfigurace parametrů a načtení datasetu (shodné s MILP notebookem)
2. Evaluátor účelové funkce dle rovnice (8)
3. Parametrizace ACO a inicializace feromonové matice
4. Heuristická informace η_ij a seznam kandidátů
5. Konstrukce řešení mravencem a aktualizace feromonů
6. Hlavní smyčka ACO a analýza výsledků

**Požadované knihovny:** pandas, numpy, openpyxl

In [1]:
import math
import random
import time
from dataclasses import dataclass
from typing import List, Dict, Tuple, Any

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

@dataclass(frozen=True)
class Config:
    
    m: int = 3
    Q: int = 13

    unit_demand: int = 1
    service_time: float = 1.0
    speed: float = 1.0

    PI_LATE: float = 1.5
    P_SKIP: float = 18.0

    # TW generátor
    tw_horizon: float = 120.0
    tw_width: float = 20.0
    tw_a_min: float = 0.0
    tw_a_max: float = 80.0

cfg = Config()
print(cfg)


Config(m=3, Q=13, unit_demand=1, service_time=1.0, speed=1.0, PI_LATE=1.5, P_SKIP=18.0, tw_horizon=120.0, tw_width=20.0, tw_a_min=0.0, tw_a_max=80.0)


## 1. Načtení datasetu a příprava vstupních dat

Načtení datového souboru small_dataset.xlsx (Příloha 1), výpočet euklidovské vzdálenostní matice d_ij, cestovních časů t_ij = d_ij (při speed = 1,0 dle rovnice 9), jednotkové poptávky q_i = 1 a generování časových oken [e_i, l_i] se seedem SEED = 42 dle tabulky 2.2 diplomové práce. Postup je shodný s MILP (Příloha 2) a HS (Příloha 3) notebooky.

In [2]:
# Načtení datasetu

# Cesta k datasetu (nutnost upravit podle aktuálního uložení datasetu)
DATA_PATH = "small_dataset.xlsx"
df = pd.read_excel(DATA_PATH)

cols_lower = [c.lower() for c in df.columns]
x_col = None
y_col = None
for cand in ["x", "coord_x", "lon", "longitude"]:
    if cand in cols_lower:
        x_col = df.columns[cols_lower.index(cand)]
        break
for cand in ["y", "coord_y", "lat", "latitude"]:
    if cand in cols_lower:
        y_col = df.columns[cols_lower.index(cand)]
        break

if x_col is None or y_col is None:
    raise ValueError(f"Nelze najít sloupce souřadnic. Sloupce v souboru: {list(df.columns)}")

coords = df[[x_col, y_col]].to_numpy(dtype=float)
n = coords.shape[0]
print("Počet uzlů n =", n)

# Matice eukleidovských vzdáleností
d = np.zeros((n, n), dtype=float)
for i in range(n):
    for j in range(n):
        if i != j:
            dx = coords[i, 0] - coords[j, 0]
            dy = coords[i, 1] - coords[j, 1]
            d[i, j] = math.hypot(dx, dy)

# Rychlost = 1 => čas = vzdálenost
t = d / cfg.speed

# Jednotkové poptávky (depot = 0)
q = np.zeros(n, dtype=int)
q[1:] = cfg.unit_demand

# Servisní časy (depot = 0)
service = np.zeros(n, dtype=float)
service[1:] = cfg.service_time

# Generátor časových oken – MUSÍ být shodný s HS/MILP
def generate_time_windows(n: int, seed: int) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)

    ready = np.zeros(n, dtype=float)  # a_i (hard)
    due = np.zeros(n, dtype=float)    # b_i (soft)

    ready[0] = 0.0
    due[0] = cfg.tw_horizon

    ready[1:] = rng.uniform(cfg.tw_a_min, cfg.tw_a_max, size=n-1)
    due[1:] = np.minimum(ready[1:] + cfg.tw_width, cfg.tw_horizon)
    return ready, due

ready_time, due_date = generate_time_windows(n, SEED)

print("TW depot:", (ready_time[0], due_date[0]))
print("TW sample customer 1:", (ready_time[1], due_date[1]))


Počet uzlů n = 30
TW depot: (0.0, 120.0)
TW sample customer 1: (61.91648388447707, 81.91648388447706)


## 2. Evaluátor účelové funkce a validace řešení

Implementace účelové funkce Z(R) dle rovnice (8) v kapitole 1.3: součet cestovních nákladů, penalizace za zpoždění (γ · T_i^k) a penalizace za neobsloužení (P_skip). Ekvivalence s fitness funkcí f(R) je prokázána v kapitole 1.6 (rovnice 46). Funkce `eval_route` vyhodnocuje jednotlivou trasu, `eval_solution` agreguje přes všechny trasy. Funkce `is_solution_valid` ověřuje kapacitní omezení, unikátnost obsluhy a konzistenci s tvrdou dolní mezí časových oken.

In [3]:
def eval_route(route: List[int]) -> Dict[str, float]:
    """
    Vyhodnocení jedné trasy:
    - travel distance
    - tardiness_sum: součet max(0, start_service - b_i) pro zákazníky
    - cap_violation: max(0, load - Q)
    """
    travel = 0.0
    tardiness_sum = 0.0
    load = sum(q[i] for i in route if i != 0)

    time_now = max(0.0, ready_time[0])
    prev = route[0]

    for node in route[1:]:
        travel += d[prev, node]

        arrival = time_now + service[prev] + t[prev, node]
        start_service = max(arrival, ready_time[node])  # hard lower bound
        tardiness_sum += max(0.0, start_service - due_date[node])  # soft upper bound

        time_now = start_service
        prev = node

    return {
        "travel": float(travel),
        "tardiness_sum": float(tardiness_sum),
        "cap_violation": float(max(0.0, load - cfg.Q)),
        "load": float(load),
    }


def eval_solution(routes: List[List[int]]) -> Dict[str, Any]:
    """
    Objective = distance + PI_LATE*tardiness + P_SKIP*unserved (+ hard cap penalty for safety)
    Pozn.: Kapacita je hard constraint – cap_violation má být 0. Penalizace je pojistka.
    """
    served = set()
    for r in routes:
        for node in r:
            if node != 0:
                served.add(node)

    customers = set(range(1, n))
    unserved = customers - served

    travel = 0.0
    tardiness_sum = 0.0
    cap_violation = 0.0
    per_route = []

    for r in routes:
        rr = eval_route(r)
        per_route.append(rr)
        travel += rr["travel"]
        tardiness_sum += rr["tardiness_sum"]
        cap_violation += rr["cap_violation"]

    tardiness_cost = cfg.PI_LATE * tardiness_sum
    skip_cost = cfg.P_SKIP * len(unserved)

    CAP_PENALTY = 1e6
    cap_cost = CAP_PENALTY * cap_violation

    total = travel + tardiness_cost + skip_cost + cap_cost

    return {
        "total": float(total),
        "travel": float(travel),
        "tardiness_sum": float(tardiness_sum),
        "tardiness_cost": float(tardiness_cost),
        "skip_cost": float(skip_cost),
        "cap_cost": float(cap_cost),
        "cap_violation": float(cap_violation),
        "served_cnt": int(len(served)),
        "unserved_cnt": int(len(unserved)),
        "routes": routes,
        "per_route": per_route,
    }


def is_solution_valid(routes: List[List[int]]) -> bool:
    # Každý zákazník max jednou
    seen = set()
    for r in routes:
        for node in r:
            if node == 0:
                continue
            if node in seen:
                return False
            seen.add(node)

    # Každá trasa začíná a končí v depu
    for r in routes:
        if len(r) < 2 or r[0] != 0 or r[-1] != 0:
            return False

    # Kapacita – hard
    for r in routes:
        if sum(q[i] for i in r if i != 0) > cfg.Q:
            return False

    # Hard lower TW (a_i): start_service musí být >= a_i (eval_route enforced by max) a stačí ověřit konzistenci simulací.
    for r in routes:
        time_now = max(0.0, ready_time[0])
        prev = r[0]
        for node in r[1:]:
            arrival = time_now + service[prev] + t[prev, node]
            start_service = max(arrival, ready_time[node])
            if start_service + 1e-9 < ready_time[node]:
                return False
            time_now = start_service
            prev = node

    return True


## 3. Parametry ACO a inicializace feromonové matice

Konfigurace parametrů ACO dle kapitoly 2.4 diplomové práce (tabulky 2.11 a 2.12):
- **N_ants:** počet mravenců v kolonii
- **N_iter:** počet iterací
- **α:** váha feromonové složky při výběru uzlu (rovnice 34)
- **β:** váha heuristické složky při výběru uzlu (rovnice 34)
- **ρ:** míra evaporace feromonů (rovnice 36)
- **Q_pher:** konstanta depozice feromonů

Feromonová matice τ_ij je inicializována hodnotou τ_0 = 1,0 pro všechny hrany i ≠ j. Konfigurace se mění mezi jednotlivými běhy přístupem ceteris paribus.

In [4]:
@dataclass
class ACOParams:
    N_ANTS: int = 50
    N_ITER: int = 1000
    ALPHA: float = 1.0
    BETA: float = 2.0
    RHO: float = 0.20
    Q_PHER: float = 1.0
    TAU0: float = 1.0
    CANDIDATE_LIST_SIZE: int = 10
    ELITIST: bool = True  # posiluj best-so-far
    EPS: float = 1e-12

aco = ACOParams()
print(aco)

# Feromonová matice (i != j)
tau = np.full((n, n), aco.TAU0, dtype=float)
np.fill_diagonal(tau, 0.0)


ACOParams(N_ANTS=50, N_ITER=1000, ALPHA=1.0, BETA=2.0, RHO=0.2, Q_PHER=1.0, TAU0=1.0, CANDIDATE_LIST_SIZE=10, ELITIST=True, EPS=1e-12)


## 4. Heuristická informace a seznam kandidátů

Implementace heuristické informace η_ij dle rovnice (35) v kapitole 1.6. Hodnota η_ij zohledňuje vzdálenost d_ij a tardiness inkrement ΔL_j, čímž směřuje konstrukci tras k uzlům s příznivými časovými charakteristikami. Funkce `append_feasible` ověřuje přípustnost připojení zákazníka na konec trasy z hlediska kapacity a tvrdé dolní meze časového okna. Seznam kandidátů je omezen na top-k uzlů seřazených dle η_ij.

In [5]:
def append_feasible(prev: int, cand: int, time_now: float, load_now: int) -> Tuple[bool, float, float]:
    """
    Zkus připojit 'cand' na konec aktuální trasy.
    Vrací:
      feasible, new_time, tardiness_increment
    """
    if cand == 0 or cand == prev:
        return False, time_now, 0.0
    if load_now + q[cand] > cfg.Q:
        return False, time_now, 0.0

    arrival = time_now + service[prev] + t[prev, cand]
    start_service = max(arrival, ready_time[cand])  # hard lower
    tard_inc = max(0.0, start_service - due_date[cand])  # soft upper
    return True, start_service, tard_inc


def heuristic_eta(prev: int, cand: int, time_now: float, load_now: int) -> float:
    """
    Heuristika ~ 1 / (distance + PI_LATE*tardiness_increment + small)
    (časová rizikovost je přímo inkrement tardiness)
    """
    feasible, _, tard_inc = append_feasible(prev, cand, time_now, load_now)
    if not feasible:
        return 0.0
    cost_like = d[prev, cand] + cfg.PI_LATE * tard_inc
    return 1.0 / (cost_like + aco.EPS)


def candidate_list(prev: int, remaining: List[int], time_now: float, load_now: int) -> List[int]:
    """
    Candidate list = top-k podle heuristiky (pro stabilitu a rychlost).
    """
    scored = []
    for cand in remaining:
        eta = heuristic_eta(prev, cand, time_now, load_now)
        if eta > 0.0:
            scored.append((eta, cand))
    scored.sort(reverse=True)  # max eta first
    return [c for _, c in scored[:aco.CANDIDATE_LIST_SIZE]]


## 5. Konstrukce řešení mravencem

Konstrukční procedura jednoho mravence dle pseudokódu v kapitole 1.6. Mravenec postupně buduje m = 3 tras: v každém kroku vybírá další uzel ze seznamu kandidátů s pravděpodobností úměrnou dle rovnice 34. Zákazníci, které žádná trasa nepojme, zůstávají neobslouženi a jsou penalizováni složkou P_skip v účelové funkci.

In [6]:
def construct_solution_one_ant() -> List[List[int]]:
    """
    Postupně konstruuj m tras:
      - start 0
      - dokud existují feasible kandidáti, připojuj uzly na konec trasy
      - po vyčerpání přejdi na další vozidlo
    Neobsloužení zákazníci zůstanou implicitně unserved.
    """
    remaining = list(range(1, n))
    random.shuffle(remaining)

    routes = []
    for k in range(cfg.m):
        route = [0]
        prev = 0
        time_now = max(0.0, ready_time[0])
        load_now = 0

        while True:
            if not remaining:
                break

            cand_list = candidate_list(prev, remaining, time_now, load_now)
            if not cand_list:
                break

            # Pravděpodobnosti dle (tau^alpha * eta^beta)
            weights = []
            for cand in cand_list:
                eta = heuristic_eta(prev, cand, time_now, load_now)
                w = (tau[prev, cand] ** aco.ALPHA) * (eta ** aco.BETA)
                weights.append(w)

            w_sum = sum(weights)
            if w_sum <= aco.EPS:
                break

            r = random.random() * w_sum
            acc = 0.0
            chosen = None
            for cand, w in zip(cand_list, weights):
                acc += w
                if acc >= r:
                    chosen = cand
                    break

            if chosen is None:
                chosen = cand_list[-1]

            feasible, new_time, _ = append_feasible(prev, chosen, time_now, load_now)
            if not feasible:
                # fallback: odstraníme z candidate listu a zkusíme znovu (jednou)
                remaining.remove(chosen)
                remaining.append(chosen)
                continue

            # commit
            route.append(chosen)
            remaining.remove(chosen)
            prev = chosen
            time_now = new_time
            load_now += int(q[chosen])

        route.append(0)
        routes.append(route)

    # Bezpečnost: validita (kapacita + uniqueness + depot)
    if not is_solution_valid(routes):
        # Pokud by se stalo (nemělo), opravíme na triviální validní řešení (jen depot-depot)
        routes = [[0, 0] for _ in range(cfg.m)]

    return routes


## 6. Aktualizace feromonové matice

Dvoustupňová aktualizace feromonů dle kapitoly 1.6:
1. **Evaporace:** τ_ij ← (1 − ρ) · τ_ij pro všechny hrany (rovnice 36)
2. **Depozice:** feromon ukládá výhradně globálně nejlepší řešení R_best (elitist strategie, řádek 33 pseudokódu); množství depozice je úměrné Q_pher / Z(R_best)

In [7]:
def evaporate_pheromone():
    global tau
    tau *= (1.0 - aco.RHO)
    np.fill_diagonal(tau, 0.0)
    tau[tau < aco.EPS] = aco.EPS


def deposit_pheromone(routes: List[List[int]], quality: float):
    """
    Deposice na hrany obsažené v řešení.
    quality = hodnota cílové funkce (nižší = lepší), depozice ~ Q_PHER / quality
    """
    global tau
    if quality <= 0:
        return
    delta = aco.Q_PHER / quality

    for r in routes:
        for i in range(len(r) - 1):
            a = r[i]
            b = r[i + 1]
            if a != b:
                tau[a, b] += delta


## 7. Hlavní smyčka ACO

Spuštění N_iter iterací algoritmu dle pseudokódu v kapitole 1.6. V každé iteraci N_ants mravenců nezávisle konstruuje řešení (sekce 5), z nichž je vybráno nejlepší iterační řešení. Po aktualizaci globálně nejlepšího řešení R_best je provedena evaporace a depozice feromonů (sekce 6). Konfigurace parametrů se mění mezi jednotlivými běhy dle tabulek 2.11 a 2.12 přístupem ceteris paribus. Seed SEED = 42 zajišťuje reprodukovatelnost stochastického běhu.

In [8]:
start_time = time.perf_counter()

best_routes = None
best_rec = None
best_found_iter = 0  # iterace, ve které bylo nalezeno konečné nejlepší řešení

for it in range(1, aco.N_ITER + 1):
    iter_best_routes = None
    iter_best_rec = None

    for a in range(aco.N_ANTS):
        routes = construct_solution_one_ant()
        rec = eval_solution(routes)

        if (iter_best_rec is None) or (rec["total"] < iter_best_rec["total"]):
            iter_best_routes = routes
            iter_best_rec = rec

    # aktualizace globálně nejlepšího řešení
    if (best_rec is None) or (iter_best_rec["total"] < best_rec["total"]):
        best_routes = iter_best_routes
        best_rec = iter_best_rec
        best_found_iter = it  # zaznamenání iterace zlepšení

    # průběžný výpis každých 100 iterací
    if it % 100 == 0 or it == 1:
        print(f"Iterace {it}/{aco.N_ITER} | best Z(R) = {best_rec['total']:.2f} | nalezeno v it. {best_found_iter}")

    # aktualizace feromonů
    evaporate_pheromone()

    if aco.ELITIST:
        deposit_pheromone(best_routes, best_rec["total"])
    else:
        deposit_pheromone(iter_best_routes, iter_best_rec["total"])

runtime = time.perf_counter() - start_time


Iterace 1/1000 | best Z(R) = 632.70 | nalezeno v it. 1
Iterace 100/1000 | best Z(R) = 221.00 | nalezeno v it. 97
Iterace 200/1000 | best Z(R) = 199.01 | nalezeno v it. 180
Iterace 300/1000 | best Z(R) = 174.68 | nalezeno v it. 237
Iterace 400/1000 | best Z(R) = 118.93 | nalezeno v it. 398
Iterace 500/1000 | best Z(R) = 118.93 | nalezeno v it. 398
Iterace 600/1000 | best Z(R) = 111.97 | nalezeno v it. 531
Iterace 700/1000 | best Z(R) = 108.85 | nalezeno v it. 618
Iterace 800/1000 | best Z(R) = 108.85 | nalezeno v it. 618
Iterace 900/1000 | best Z(R) = 108.85 | nalezeno v it. 618
Iterace 1000/1000 | best Z(R) = 108.85 | nalezeno v it. 618


## 8. Souhrnné výsledky

Rozklad účelové funkce Z(R) na jednotlivé složky dle rovnice (8): cestovní náklady dist(R), penalizace za zpoždění γ · tard(R) a penalizace za neobsloužení P_skip · |C \ served(R)|. Výpis tras jednotlivých vozidel a počtu obsloužených zákazníků pro srovnání s ostatními metodami (kapitola 2.5).

In [9]:
rec = best_rec

print("ŘEŠENÍ DOKONČENO (ACO)")
print(f"Runtime: {runtime:.15f} s")
print(f"Best found in iteration: {best_found_iter} / {aco.N_ITER}")
print(f"Objective: {rec['total']:.6f}")
print()

print("Objective breakdown (EXACT):")
print("distance     =", rec["travel"])
print(f"{cfg.PI_LATE}*tardiness=", cfg.PI_LATE * rec["tardiness_sum"])
print(f"{cfg.P_SKIP}*unserved  =", rec["skip_cost"])
print()

print("Objective breakdown (EXACT):")
print("total         =", rec["total"])
print("distance      =", rec["travel"])
print(
    f"{cfg.PI_LATE}*tardiness = {rec['tardiness_cost']:.6f}  "
    f"(tardiness_sum = {rec['tardiness_sum']:.6f})"
)
print(
    f"{cfg.P_SKIP}*unserved   = {rec['skip_cost']:.6f}  "
    f"(unserved = {rec['unserved_cnt']})"
)
print("cap_violation =", rec["cap_violation"])
print()

print("Served customers:", rec["served_cnt"], "/", (n - 1))
print("Routes:")
for k, r in enumerate(rec["routes"], start=1):
    print(f"Vehicle {k}: {r}")

ŘEŠENÍ DOKONČENO (ACO)
Runtime: 172.425853699969593 s
Best found in iteration: 618 / 1000
Objective: 108.849385

Objective breakdown (EXACT):
distance     = 104.97148309810184
1.5*tardiness= 3.8779019201036853
18.0*unserved  = 0.0

Objective breakdown (EXACT):
total         = 108.84938501820552
distance      = 104.97148309810184
1.5*tardiness = 3.877902  (tardiness_sum = 2.585268)
18.0*unserved   = 0.000000  (unserved = 0)
cap_violation = 0.0

Served customers: 29 / 29
Routes:
Vehicle 1: [0, 28, 20, 4, 13, 25, 21, 3, 6, 19, 0]
Vehicle 2: [0, 9, 29, 22, 15, 16, 17, 8, 24, 12, 7, 0]
Vehicle 3: [0, 26, 5, 18, 2, 27, 10, 11, 23, 14, 1, 0]
